# TurboQuant — KV Cache Compression Demo

**TurboQuant** compresses the KV cache during LLM inference using random orthogonal rotation + Lloyd-Max quantization.  
No fine-tuning. No calibration. Drop-in replacement for `DynamicCache`.

- **3–4× memory reduction** on KV cache at 4-bit
- Works with any HuggingFace model using standard `DynamicCache`
- Runs on GPU (recommended) or CPU

📄 Paper: [TurboQuant (ICLR 2026)](https://arxiv.org/abs/2504.19874)  
📦 Package: [turboquant-serve](https://github.com/sammyboi1801/turboquant-serve)

---
### How to use this notebook
1. Run **Section 1** to install dependencies  ← restart kernel after this
2. Run **Section 2** to pick your model
3. Run **Section 3** to load it
4. **Section 4** — single inference with KV stats
5. **Section 5** — TQ vs Baseline side-by-side comparison
6. **Section 6** — multi-turn chat
7. **Section 7** — long context memory test
8. **Section 8** *(optional)* — run as OpenAI-compatible server + web UI

## Section 1 — Install Dependencies

In [ ]:
# Install latest transformers (required for new models like Gemma 4, Llama 4)
# and turboquant-serve
import subprocess, sys

subprocess.run([sys.executable, "-m", "pip", "install",
    "git+https://github.com/huggingface/transformers.git", "-q"])
subprocess.run([sys.executable, "-m", "pip", "install",
    "turboquant-serve", "-q"])

print("Done. RESTART THE RUNTIME/KERNEL before continuing.")
print("Colab: Runtime → Restart session")
print("Kaggle: Session → Restart")

## Section 2 — Choose Your Model

Edit the `MODEL_ID` below. Pick from the presets or enter any HuggingFace repo ID.

**GPU required for models ≥ 3B. CPU works for tiny models only.**

| Model | Size | Login? | Notes |
|-------|------|--------|-------|
| `google/gemma-4-E4B-it` | 8B MoE | yes | Best quality on T4, tested ✅ |
| `google/gemma-4-E2B-it` | 5B MoE | yes | Lighter, use key_bits=8 |
| `deepseek-ai/DeepSeek-R1-Distill-Qwen-14B` | 14B | no | Reasoning model, great KV demo |
| `deepseek-ai/DeepSeek-R1-Distill-Qwen-7B` | 7B | no | Reasoning, faster |
| `Qwen/Qwen2.5-14B-Instruct` | 14B | no | Great quality, no login |
| `Qwen/Qwen2.5-7B-Instruct` | 7B | no | Fast, reliable |
| `Qwen/Qwen3.5-9B` | 9B | no | Latest Qwen |
| `microsoft/Phi-4` | 14B | no | Punches above weight |
| `meta-llama/Llama-3.1-8B-Instruct` | 8B | yes | Llama login required |

**Gated models** (Gemma, Llama): run the HF login cell below first.

In [ ]:
# ── Model config — EDIT THIS ──────────────────────────────────────────────────

MODEL_ID   = "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B"  # change to any model above

KEY_BITS   = 8   # 8 for models < 13B  |  4 for models ≥ 13B
                 # Rule: head_dim=128 (most 7B/8B models) → needs 8-bit keys
                 #       head_dim=256 (13B+, Gemma 4 E4B)  → 4-bit keys fine
VALUE_BITS = 4   # 4 is fine for all sizes
MAX_TOKENS = 1500  # reasoning models (DeepSeek R1) need 1000+ for think chain

# ─────────────────────────────────────────────────────────────────────────────
print(f"Model:       {MODEL_ID}")
print(f"Key bits:    {KEY_BITS}")
print(f"Value bits:  {VALUE_BITS}")
print(f"Max tokens:  {MAX_TOKENS}")
print(f"Compression: ~{16 / ((KEY_BITS + VALUE_BITS) / 2):.1f}x vs bf16")

In [ ]:
# ── HuggingFace login — only needed for gated models (Gemma, Llama) ──────────
# Skip this cell if using Qwen, DeepSeek, Phi — they don't require login

HF_TOKEN = ""  # paste your token from https://huggingface.co/settings/tokens

if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN)
    print("Logged in to HuggingFace")
else:
    # Try Kaggle secrets
    try:
        from kaggle_secrets import UserSecretsClient
        import os
        HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
        from huggingface_hub import login
        login(token=HF_TOKEN)
        print("Logged in via Kaggle secret")
    except Exception:
        print("No HF token set — skipping login (fine for public models)")

## Section 3 — Load Model

In [ ]:
import torch
import gc
from transformers import DynamicCache
from turboquant import load_model, TurboQuantCache

# Load model — auto NF4 on GPU < 16 GB, float32 on CPU
model, tokenizer = load_model(MODEL_ID)

# Show GPU status
if torch.cuda.is_available():
    free_b, total_b = torch.cuda.mem_get_info(0)
    used_b = total_b - free_b
    print(f"\nGPU:  {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {used_b/1e9:.1f} GB used / {total_b/1e9:.1f} GB total ({free_b/1e9:.1f} GB free)")
else:
    print("\nRunning on CPU")

## Section 4 — Single Inference with TurboQuant

In [ ]:
import time

def infer(prompt, key_bits=KEY_BITS, value_bits=VALUE_BITS, max_tokens=MAX_TOKENS):
    """Run inference with TurboQuantCache and print KV compression stats."""
    msgs = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    cache = TurboQuantCache(key_bits=key_bits, value_bits=value_bits)
    t0 = time.perf_counter()
    with torch.inference_mode():
        out = model.generate(
            **inputs, past_key_values=cache,
            max_new_tokens=max_tokens, do_sample=False,
            repetition_penalty=1.1,
        )
    elapsed = time.perf_counter() - t0

    n_tokens = out.shape[1] - inputs.input_ids.shape[1]
    reply = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    stats = cache.compression_stats()

    print(reply)
    print(f"\n{'─'*60}")
    print(f"Tokens:      {n_tokens}  ({n_tokens/elapsed:.1f} tok/s)")
    print(f"KV cache:    {stats['compressed_MB']:.2f} MB  (TurboQuant)")
    print(f"Baseline:    {stats['baseline_MB']:.2f} MB  (DynamicCache bf16)")
    print(f"Saved:       {stats['baseline_MB'] - stats['compressed_MB']:.2f} MB")
    print(f"Compression: {stats['ratio']:.2f}x")
    return reply

In [ ]:
# ── Run inference — change the prompt ────────────────────────────────────────

infer("Explain how GPS works")

## Section 5 — TurboQuant vs Baseline Comparison

Runs the **same prompt** with TurboQuant and standard DynamicCache back-to-back.  
Shows output quality + actual memory used by each.

In [ ]:
def compare(prompt, key_bits=KEY_BITS, value_bits=VALUE_BITS, max_tokens=MAX_TOKENS):
    """Compare TurboQuant vs DynamicCache — same prompt, side by side."""
    msgs = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    def vram_mb():
        return torch.cuda.memory_allocated() / 1e6 if torch.cuda.is_available() else 0

    gen_kwargs = dict(
        max_new_tokens=max_tokens, do_sample=False, repetition_penalty=1.1
    )

    # ── TurboQuant ────────────────────────────────────────────────────────────
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    v0 = vram_mb()
    tq_cache = TurboQuantCache(key_bits=key_bits, value_bits=value_bits)
    t0 = time.perf_counter()
    with torch.inference_mode():
        out = model.generate(**inputs, past_key_values=tq_cache, **gen_kwargs)
    tq_time = time.perf_counter() - t0
    tq_tokens = out.shape[1] - inputs.input_ids.shape[1]
    tq_reply = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    tq_stats = tq_cache.compression_stats()
    tq_vram = vram_mb() - v0
    del tq_cache; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

    # ── Baseline DynamicCache ─────────────────────────────────────────────────
    v0 = vram_mb()
    base_cache = DynamicCache()
    t0 = time.perf_counter()
    try:
        with torch.inference_mode():
            out = model.generate(**inputs, past_key_values=base_cache, **gen_kwargs)
        base_time = time.perf_counter() - t0
        base_tokens = out.shape[1] - inputs.input_ids.shape[1]
        base_reply = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
        base_vram = vram_mb() - v0
        base_oom = False
    except torch.cuda.OutOfMemoryError:
        base_reply = "[OUT OF MEMORY — TurboQuant prevented this OOM]"
        base_tokens, base_time, base_vram = 0, 0, 0
        base_oom = True
    del base_cache; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

    # ── Print results ─────────────────────────────────────────────────────────
    W = 62
    print(f"\n{'═'*W}")
    print(f"  Prompt: {prompt[:W-10]}")
    print(f"{'═'*W}")

    print(f"\n{'─'*W}")
    print("  TurboQuant output:")
    print(f"{'─'*W}")
    print(tq_reply)

    print(f"\n{'─'*W}")
    print("  Baseline (DynamicCache) output:")
    print(f"{'─'*W}")
    print(base_reply)

    print(f"\n{'═'*W}")
    print(f"  {'Metric':<28} {'TurboQuant':>14} {'Baseline':>14}")
    print(f"  {'─'*28} {'─'*14} {'─'*14}")
    print(f"  {'KV cache (MB)':<28} {tq_stats['compressed_MB']:>14.2f} {tq_stats['baseline_MB']:>14.2f}")
    print(f"  {'VRAM delta (MB)':<28} {tq_vram:>14.1f} {base_vram:>14.1f}")
    print(f"  {'Speed (tok/s)':<28} {tq_tokens/max(tq_time,1e-6):>14.1f} {base_tokens/max(base_time,1e-6) if not base_oom else 'OOM':>14}")
    print(f"  {'Compression ratio':<28} {tq_stats['ratio']:>14.2f}x {'1.00x':>14}")
    print(f"  {'Memory saved (MB)':<28} {tq_stats['baseline_MB'] - tq_stats['compressed_MB']:>14.2f} {'—':>14}")
    print(f"{'═'*W}")

In [ ]:
# ── Run comparison — change the prompt ───────────────────────────────────────

compare("Explain the theory of relativity in simple terms")

## Section 6 — Multi-turn Chat

Stateless like the OpenAI API — full history sent each turn.  
TurboQuant's benefit: longer conversations before OOM.

In [ ]:
conversation_history = []

def chat(user_input, max_tokens=MAX_TOKENS):
    """Multi-turn chat with TurboQuant. Sends full history each turn."""
    global conversation_history
    conversation_history.append({"role": "user", "content": user_input})

    text = tokenizer.apply_chat_template(
        conversation_history, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    cache = TurboQuantCache(key_bits=KEY_BITS, value_bits=VALUE_BITS)
    with torch.inference_mode():
        out = model.generate(
            **inputs, past_key_values=cache,
            max_new_tokens=max_tokens, do_sample=False,
            repetition_penalty=1.1,
        )

    reply = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    conversation_history.append({"role": "assistant", "content": reply})

    stats = cache.compression_stats()
    print(f"You: {user_input}")
    print(f"\nAssistant: {reply}")
    print(f"\nKV: {stats['compressed_MB']:.2f} MB  |  baseline: {stats['baseline_MB']:.2f} MB  |  {stats['ratio']:.2f}x  |  turn {len(conversation_history)//2}")
    print(f"{'─'*60}")
    return reply

def reset_chat():
    global conversation_history
    conversation_history = []
    print("Conversation reset.")

In [ ]:
# ── Multi-turn conversation — add more chat() calls to continue ───────────────

reset_chat()
chat("What is quantum entanglement?")
chat("Give me a simple analogy for it")
chat("How is it used in quantum computing?")

## Section 7 — Long Context Memory Test

Shows TurboQuant's advantage at **long context** — the longer the prompt, the more memory it saves.  
Also handles OOM gracefully: if baseline runs out of memory, TurboQuant still works.

In [ ]:
FILLER = (
    "The development of large language models has accelerated significantly "
    "over the past few years. Researchers have proposed numerous techniques "
    "to improve efficiency, including quantization, pruning, and distillation. "
)
NEEDLE = "The secret password is QUANTUM-ALPHA-9371."
QUESTION = "What is the secret password mentioned in the document?"

def make_prompt(context_tokens, depth=0.5):
    filler_len = len(tokenizer.encode(FILLER))
    repeats = (int(context_tokens * 0.9) // filler_len) + 1
    words = (FILLER * repeats).split()
    insert_at = int(len(words) * depth)
    words.insert(insert_at, f" {NEEDLE} ")
    doc = " ".join(words)
    return f"Read this document:\n\n{doc}\n\nQuestion: {QUESTION}\nAnswer:"

def run_memory_test(context_lengths=[512, 1024, 2048, 4096]):
    print(f"\n{'─'*65}")
    print(f"  {'Context':>8}  {'TQ KV (MB)':>12}  {'Base KV (MB)':>14}  {'Ratio':>7}  {'Needle':>8}")
    print(f"{'─'*65}")

    for ctx_len in context_lengths:
        prompt = make_prompt(ctx_len)
        inputs = tokenizer(prompt, return_tensors="pt",
                           max_length=ctx_len+100, truncation=True).to(model.device)
        actual_len = inputs.input_ids.shape[1]

        try:
            tq_cache = TurboQuantCache(key_bits=KEY_BITS, value_bits=VALUE_BITS)
            with torch.inference_mode():
                out = model.generate(**inputs, past_key_values=tq_cache,
                                     max_new_tokens=30, do_sample=False)
            answer = tokenizer.decode(out[0][actual_len:], skip_special_tokens=True).strip()
            found = "QUANTUM-ALPHA-9371" in answer or "quantum" in answer.lower()
            stats = tq_cache.compression_stats()
            result = "✓ FOUND" if found else "✗ MISSED"
            print(f"  {actual_len:>8,}  {stats['compressed_MB']:>12.2f}  {stats['baseline_MB']:>14.2f}  {stats['ratio']:>6.2f}x  {result:>8}")
            del tq_cache
        except torch.cuda.OutOfMemoryError:
            print(f"  {actual_len:>8,}  {'OOM':>12}  {'OOM':>14}  {'—':>7}  {'OOM':>8}")
            if torch.cuda.is_available(): torch.cuda.empty_cache()

        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

    print(f"{'─'*65}")

run_memory_test([512, 1024, 2048, 4096])

## Section 8 (Optional) — OpenAI-Compatible Server + Web UI

> **Skip this section for normal use.** Sections 4–7 cover everything via direct Python.  
> Use this only if you want the web UI or to expose an OpenAI-compatible API endpoint.

Starts `tq-serve` in the background and gives you a public URL with:
- `/ui` — chat, compare, live GPU stats
- `/v1/chat/completions` — OpenAI-compatible API
- `/v1/compare` — TQ vs baseline in one call

**On Colab:** uses built-in proxy (no setup)  
**On Kaggle:** uses cloudflared tunnel

In [ ]:
import subprocess, threading, time, re, os, requests

SERVER_PORT = 8000

# ── Start server ──────────────────────────────────────────────────────────────
def run_server():
    with open("/tmp/tq_server.log", "w") as log:
        subprocess.run([
            "tq-serve", "--model", MODEL_ID,
            "--key-bits", str(KEY_BITS),
            "--value-bits", str(VALUE_BITS),
            "--port", str(SERVER_PORT),
        ], stdout=log, stderr=log)

threading.Thread(target=run_server, daemon=True).start()

# ── Wait until ready ──────────────────────────────────────────────────────────
print("Waiting for server (check /tmp/tq_server.log if this hangs)...")
for i in range(60):
    try:
        r = requests.get(f"http://localhost:{SERVER_PORT}/health", timeout=3)
        if r.json().get("warmed_up"):
            print(f"Server ready after {i*5}s")
            break
    except Exception:
        pass
    time.sleep(5)

# ── Get public URL (Colab or Kaggle) ─────────────────────────────────────────
def get_url(port):
    try:
        from google.colab.output import eval_js
        url = eval_js(f"google.colab.kernel.proxyPort({port})")
        print(f"\nWeb UI:  {url}/ui")
        print(f"Health:  {url}/health")
        return url
    except ImportError:
        pass
    # Kaggle — cloudflared
    if not os.path.exists("./cloudflared"):
        subprocess.run(["wget", "-q",
            "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
            "-O", "cloudflared"])
        subprocess.run(["chmod", "+x", "cloudflared"])
    proc = subprocess.Popen(["./cloudflared", "tunnel", "--url", f"http://localhost:{port}"],
                            stderr=subprocess.PIPE)
    time.sleep(8)
    for _ in range(30):
        line = proc.stderr.readline().decode()
        m = re.search(r'https://[a-z0-9\-]+\.trycloudflare\.com', line)
        if m:
            print(f"\nWeb UI:  {m.group(0)}/ui")
            print(f"Health:  {m.group(0)}/health")
            return m.group(0)
        time.sleep(1)

get_url(SERVER_PORT)